# Chapter 2: Modelling

## § 1. From the brief to an ML task

| Component | Description |
| :--- | :--- |
| **What decision does the prediction feed?** | Help the SOC Analyst decide which login alerts to block automatically or investigate for high risk. |
| **Task type** | Binary Classification (predicting whether a login attempt is legitimate or an anomalous threat). |
| **Target (exact column)** | `Is_Anomaly` |
| **Grain** | One individual login attempt per user session. |
| **Prediction moment & horizon** | Evaluated in real-time or via daily batch processing using session metadata at the exact moment of login. |
| **Available features** | Session metadata, device type, browser, IP address, and historical login patterns (excluding future or post-event data). |
| **Today's rule (Baseline)** | Current static IP blocking rules and fixed failed-attempt thresholds. |
| **Model metric** | ROC-AUC / PR-AUC, balancing the cost of false positives vs. false negatives. |

> **What this means for the stakeholder:** This model translates technical log data into a clear, actionable decision for the security team, allowing us to block automated attacks without disrupting legitimate users.

## § 2. Data and quality

In this section, we load the dataset using a relative path, inspect its dimensions, and verify data quality.

In [ ]:
import pandas as pd
import numpy as np

# Relative path loading ensuring correct repository structure
df = pd.read_csv('data/login_anomaly_dataset.csv')

# Display basic dataset info
print(f"Dataset shape: {df.shape}")
display(df.head())

> **What this means for the stakeholder:** Validating data quality early ensures our threat detection model trains on clean, reliable session records without blind spots or corrupted features.

## § 3. What the exploration showed

We investigate the distribution of login anomalies. Security logs typically show high class imbalance, as malicious login attempts represent a small fraction of overall network traffic.

In [ ]:
import plotly.express as px

# Target distribution plot
fig_target = px.histogram(df, x='Is_Anomaly', title='Distribution of Login Anomalies (Target Variable)',
                          labels={'Is_Anomaly': 'Anomaly Status (0 = Normal, 1 = Anomaly)', 'count': 'Number of Sessions'})
fig_target.show()

> **What this means for the stakeholder:** Visualizing the imbalance between normal and anomalous logins helps the security team understand the scale of threats we face daily, justifying the need for automated anomaly detection.

## § 4. Features and leakage check

To ensure robust performance and prevent data leakage, we evaluate each feature against a strict operational constraint: **'Would I realistically have this value at the exact moment of login evaluation?'**

- **Session Metadata & Geolocation:** Included, as they are available during handshake.
- **Post-Authentication Flags:** Excluded to avoid target leakage.

> **What this means for the stakeholder:** Rigorous feature auditing ensures our model makes predictions using only information available in real-time, preventing artificial inflation of test scores and guaranteeing reliability in production.

## § 5. Split and baseline

Since our data contains temporal records, we must split the dataset respecting time (training on past logs and testing on future logs) to simulate real-world deployment[cite: 18]. We establish a naive baseline and today's static rule to measure real improvement.

In [ ]:
# Baseline evaluation logic placeholder
from sklearn.model_selection import train_test_split

# Simple chronological simulation or train-test split with fixed random_state for reproducibility
if 'Is_Anomaly' in df.columns:
    X = df.drop(columns=['Is_Anomaly'])
    y = df['Is_Anomaly']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

> **What this means for the stakeholder:** Establishing a clear baseline using past security logs allows us to prove mathematically how much better our machine learning model performs compared to traditional static rules.